# Natural Shape-Space Analysis

This notebook analyzes the **natural evaluation datasets**:

1. **BabyLand72**
2. **InfantFace**

It is designed for **shape-space analysis** and **domain-gap diagnostics** using the **synthetic PCA prior(s)** only.

Important methodological rule:

- **Do not fit PCA on BabyLand72 or InfantFace.**
- PCA fitting must remain based on the **synthetic train set**.
- Natural datasets are projected into the synthetic PCA space for reconstruction-error and score-space analysis.

## How to run

1. Configure the dataset and prior paths in the **Configuration** section.
2. Run the notebook top to bottom.
3. All figures and tables will be saved under the configured output directory.

Recommended usage:

- Use **BabyLand72-only 72-landmark analysis** where 72-point natural detail matters.
- Use **first-68-landmark analysis** for direct BabyLand72 vs InfantFace comparison.
- Use the **synthetic PCA prior(s)** only for projection/reconstruction diagnostics.

In [6]:
from __future__ import annotations

import json
import math
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
    stats = None

import torch

REPO_ROOT = Path("/Users/jocareher/Library/CloudStorage/OneDrive-Personal/Educacion/PhD_UPF_2023/landmarks_detection/")#Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.utils.natural_labels import (
    NATURAL_CLASS_ID_TO_ORIENTATION,
    parse_natural_landmark_label,
)
from scripts.engine.pca_shape_prior import load_pca_shape_prior

In [30]:
# =============================
# Configuration
# =============================

BABYLAND72_ROOT = Path('/Users/jocareher/Documents/baby_face_72')
INFANTFACE_ROOT = Path('/Users/jocareher/Documents/infanface_adapted/all')

# Required synthetic PCA prior (.pt) built from the synthetic train split only.
SYNTHETIC_PCA_PRIOR_PATH = Path('/Users/jocareher/Documents/synthetic_lmks_vis_dataset/class_conditioned_pca_prior_k32.pt')

# Optional additional resources for richer synthetic-vs-natural comparison.
SYNTHETIC_TRAIN_ROOT: Optional[Path] = Path("/Users/jocareher/Documents/synthetic_lmks_vis_dataset/train")
SYNTHETIC_REFERENCE_TABLES_DIR: Optional[Path] = Path("/Users/jocareher/Documents/synthetic_lmks_vis_dataset/shape_analysis_outputs/tables")

OUTPUT_ROOT = Path('/Users/jocareher/Documents/natural_shape_analysis_outputs')
TABLES_DIR = OUTPUT_ROOT / 'tables'
FIGURES_DIR = OUTPUT_ROOT / 'figures'
OUTLIERS_DIR = OUTPUT_ROOT / 'outliers'
for directory in [TABLES_DIR, FIGURES_DIR, OUTLIERS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BABYLAND72_NUM_LANDMARKS = 72
INFANTFACE_NUM_LANDMARKS = 68
COMPARISON_NUM_LANDMARKS = 68

CLASS_NAMES = dict(NATURAL_CLASS_ID_TO_ORIENTATION)
CLASS_ORDER = [CLASS_NAMES[idx] for idx in sorted(CLASS_NAMES)]
CLASS_COLORS = {
    'left': '#1f77b4',
    'quarter_left': '#5dade2',
    'frontal': '#2ca02c',
    'quarter_right': '#f5b041',
    'right': '#d62728',
}
DATASET_COLORS = {
    'BabyLand72': '#4c78a8',
    'InfantFace': '#f58518',
    'SyntheticTrain': '#54a24b',
}

ANATOMICAL_GROUP_COLORS = {
    'face_contour': '#4e79a7',
    'right_eyebrow': '#f28e2b',
    'left_eyebrow': '#e15759',
    'nose_bridge': '#76b7b2',
    'nose_base': '#59a14f',
    'right_eye': '#edc948',
    'left_eye': '#b07aa1',
    'outer_lip': '#ff9da7',
    'inner_lip': '#9c755f',
    'under_lip': '#bab0ab',
    'upper_chin': '#8cd17d',
    'left_chin': '#499894',
    'right_chin': '#e15759',
}

SAVE_PDF = True
FIG_DPI = 250
TOP_K_OUTLIERS = 20
MAX_PCS_TO_PLOT = 5

# When comparing BabyLand72 and InfantFace directly, only the first 68 landmarks are used.
ENABLE_72_POINT_BABYLAND72_ANALYSIS = True

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (8, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16,
})

## Landmark and anatomical mappings

This section defines the anatomical groups and landmark labels used throughout the notebook.

- The standard 68-landmark regions are reused for direct cross-dataset comparison.
- BabyLand72-specific landmarks 69-72 are kept available for BabyLand72-only 72-point analysis.

In [15]:
LANDMARK_LABELS_72 = [
    *(f'face_contour_{idx}' for idx in range(1, 18)),
    *(f'right_eyebrow_{idx}' for idx in range(18, 23)),
    *(f'left_eyebrow_{idx}' for idx in range(23, 28)),
    *(f'nose_bridge_{idx}' for idx in range(28, 32)),
    *(f'nose_base_{idx}' for idx in range(32, 37)),
    *(f'right_eye_{idx}' for idx in range(37, 43)),
    *(f'left_eye_{idx}' for idx in range(43, 49)),
    *(f'outer_lip_{idx}' for idx in range(49, 61)),
    *(f'inner_lip_{idx}' for idx in range(61, 69)),
    'under_lip_69',
    'upper_chin_70',
    'left_chin_71',
    'right_chin_72',
]

LANDMARK_CONNECTIONS_72 = {
    'face_contour': list(range(0, 17)),
    'right_eyebrow': list(range(17, 22)),
    'left_eyebrow': list(range(22, 27)),
    'nose_bridge': list(range(27, 31)),
    'nose_base': list(range(31, 36)),
    'right_eye': [36, 37, 38, 39, 40, 41, 36],
    'left_eye': [42, 43, 44, 45, 46, 47, 42],
    'outer_lip': list(range(48, 60)) + [48],
    'inner_lip': list(range(60, 68)) + [60],
    'under_lip': [68],
    'upper_chin': [69],
    'left_chin': [70],
    'right_chin': [71],
}


def get_anatomical_group(landmark_idx: int) -> str:
    if 0 <= landmark_idx <= 16:
        return 'face_contour'
    if 17 <= landmark_idx <= 21:
        return 'right_eyebrow'
    if 22 <= landmark_idx <= 26:
        return 'left_eyebrow'
    if 27 <= landmark_idx <= 30:
        return 'nose_bridge'
    if 31 <= landmark_idx <= 35:
        return 'nose_base'
    if 36 <= landmark_idx <= 41:
        return 'right_eye'
    if 42 <= landmark_idx <= 47:
        return 'left_eye'
    if 48 <= landmark_idx <= 59:
        return 'outer_lip'
    if 60 <= landmark_idx <= 67:
        return 'inner_lip'
    if landmark_idx == 68:
        return 'under_lip'
    if landmark_idx == 69:
        return 'upper_chin'
    if landmark_idx == 70:
        return 'left_chin'
    if landmark_idx == 71:
        return 'right_chin'
    raise ValueError(f'Unsupported landmark_idx={landmark_idx}.')


def get_anatomical_label(landmark_idx: int) -> str:
    group = get_anatomical_group(landmark_idx)
    if landmark_idx < 68:
        return f'{group}_{landmark_idx + 1}'
    return LANDMARK_LABELS_72[landmark_idx]


LANDMARK_METADATA_72 = pd.DataFrame({
    'landmark_idx': np.arange(72, dtype=int),
    'landmark_number': np.arange(1, 73, dtype=int),
    'anatomical_group': [get_anatomical_group(idx) for idx in range(72)],
    'anatomical_label': [get_anatomical_label(idx) for idx in range(72)],
})
LANDMARK_METADATA_68 = LANDMARK_METADATA_72.iloc[:68].copy().reset_index(drop=True)
LANDMARK_METADATA_72.head()

,landmark_idx,landmark_number,anatomical_group,anatomical_label
0,0,1,face_contour,face_contour_1
1,1,2,face_contour,face_contour_2
2,2,3,face_contour,face_contour_3
3,3,4,face_contour,face_contour_4
4,4,5,face_contour,face_contour_5


## Label parsers

These parsers are robust to the natural label formats used in the current repository:

- **BabyLand72**: `class_idx + 72 rows of x y v`
- **InfantFace**: `class_idx + 68 rows of x y`

Each loaded sample keeps:

- `image_id`
- `class_idx`
- `orientation`
- `landmarks`
- `visibility` if available
- `dataset_name`

In [16]:
VALID_IMAGE_SUFFIXES = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.JPG', '.JPEG', '.PNG', '.BMP')


@dataclass
class NaturalShapeSample:
    dataset_name: str
    image_id: str
    label_path: Path
    image_path: Optional[Path]
    class_idx: int
    orientation: str
    landmarks: np.ndarray
    visibility: Optional[np.ndarray]
    num_landmarks: int


def resolve_label_dir(root_path: Path) -> Path:
    root_path = Path(root_path)
    labels_dir = root_path / 'labels'
    if labels_dir.is_dir() and any(labels_dir.glob('*.txt')):
        return labels_dir
    if root_path.is_dir() and any(root_path.glob('*.txt')):
        return root_path
    raise FileNotFoundError(f'Could not locate label txt files under {root_path}.')


def resolve_images_dir(root_path: Path) -> Optional[Path]:
    images_dir = Path(root_path) / 'images'
    return images_dir if images_dir.is_dir() else None


def find_image_for_stem(images_dir: Optional[Path], stem: str) -> Optional[Path]:
    if images_dir is None:
        return None
    for suffix in VALID_IMAGE_SUFFIXES:
        candidate = images_dir / f'{stem}{suffix}'
        if candidate.exists():
            return candidate
    return None


def parse_babyland72_label(label_path: Path) -> tuple[int, str, np.ndarray, np.ndarray]:
    parsed = parse_natural_landmark_label(label_path=label_path, expected_num_landmarks=BABYLAND72_NUM_LANDMARKS)
    class_idx = -1 if parsed.class_idx is None else int(parsed.class_idx)
    return class_idx, parsed.orientation, parsed.landmarks.astype(np.float64), parsed.visibility.astype(np.float64)


def parse_infantface_label(label_path: Path) -> tuple[int, str, np.ndarray]:
    lines = [line.strip() for line in Path(label_path).read_text(encoding='utf-8').splitlines() if line.strip()]
    if not lines:
        raise ValueError(f'Empty InfantFace label file: {label_path}')
    first_tokens = lines[0].split()
    if len(first_tokens) != 1:
        raise ValueError(f'Expected class_idx header in first line of {label_path}, got: {lines[0]!r}')
    class_idx = int(float(first_tokens[0]))
    if class_idx not in CLASS_NAMES:
        raise ValueError(f'Unsupported InfantFace class_idx={class_idx} in {label_path}.')
    rows = []
    for line_number, raw_line in enumerate(lines[1:], start=2):
        tokens = raw_line.split()
        if len(tokens) != 2:
            raise ValueError(
                f'Invalid InfantFace landmark row in {label_path} at line {line_number}. '
                f"Expected 'x y', got: {raw_line!r}."
            )
        rows.append([float(tokens[0]), float(tokens[1])])
    landmarks = np.asarray(rows, dtype=np.float64)
    expected_shape = (INFANTFACE_NUM_LANDMARKS, 2)
    if landmarks.shape != expected_shape:
        raise ValueError(f'Invalid InfantFace shape in {label_path}. Expected {expected_shape}, got {landmarks.shape}.')
    return class_idx, CLASS_NAMES[class_idx], landmarks


def load_babyland72_samples(root_path: Path) -> list[NaturalShapeSample]:
    labels_dir = resolve_label_dir(root_path)
    images_dir = resolve_images_dir(root_path)
    samples: list[NaturalShapeSample] = []
    for label_path in sorted(labels_dir.glob('*.txt')):
        class_idx, orientation, landmarks, visibility = parse_babyland72_label(label_path)
        samples.append(
            NaturalShapeSample(
                dataset_name='BabyLand72',
                image_id=label_path.stem,
                label_path=label_path,
                image_path=find_image_for_stem(images_dir, label_path.stem),
                class_idx=class_idx,
                orientation=orientation,
                landmarks=landmarks,
                visibility=visibility,
                num_landmarks=landmarks.shape[0],
            )
        )
    return samples


def load_infantface_samples(root_path: Path) -> list[NaturalShapeSample]:
    labels_dir = resolve_label_dir(root_path)
    images_dir = resolve_images_dir(root_path)
    samples: list[NaturalShapeSample] = []
    for label_path in sorted(labels_dir.glob('*.txt')):
        class_idx, orientation, landmarks = parse_infantface_label(label_path)
        samples.append(
            NaturalShapeSample(
                dataset_name='InfantFace',
                image_id=label_path.stem,
                label_path=label_path,
                image_path=find_image_for_stem(images_dir, label_path.stem),
                class_idx=class_idx,
                orientation=orientation,
                landmarks=landmarks,
                visibility=None,
                num_landmarks=landmarks.shape[0],
            )
        )
    return samples

## Utility helpers

These helpers keep output naming, geometric masking, PCA projection, and plotting behavior consistent across all sections.

In [17]:
def save_table(df: pd.DataFrame, filename: str) -> Path:
    output_path = TABLES_DIR / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    return output_path


def save_figure(fig: plt.Figure, relative_stem: str) -> dict[str, Path]:
    png_path = FIGURES_DIR / f'{relative_stem}.png'
    png_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(png_path, dpi=FIG_DPI, bbox_inches='tight')
    saved = {'png': png_path}
    if SAVE_PDF:
        pdf_path = FIGURES_DIR / f'{relative_stem}.pdf'
        fig.savefig(pdf_path, bbox_inches='tight')
        saved['pdf'] = pdf_path
    return saved


def trim_sample_to_landmarks(sample: NaturalShapeSample, num_landmarks: int) -> NaturalShapeSample:
    visibility = None if sample.visibility is None else sample.visibility[:num_landmarks].copy()
    return NaturalShapeSample(
        dataset_name=sample.dataset_name,
        image_id=sample.image_id,
        label_path=sample.label_path,
        image_path=sample.image_path,
        class_idx=sample.class_idx,
        orientation=sample.orientation,
        landmarks=sample.landmarks[:num_landmarks].copy(),
        visibility=visibility,
        num_landmarks=num_landmarks,
    )


def sample_valid_mask(sample: NaturalShapeSample, num_landmarks: Optional[int] = None) -> np.ndarray:
    num_landmarks = sample.num_landmarks if num_landmarks is None else num_landmarks
    finite_mask = np.isfinite(sample.landmarks[:num_landmarks]).all(axis=1)
    if sample.visibility is None:
        return finite_mask
    return finite_mask & (sample.visibility[:num_landmarks] == 1)


def compute_bbox_stats(coords: np.ndarray) -> dict[str, float]:
    finite_mask = np.isfinite(coords).all(axis=1)
    valid_coords = coords[finite_mask]
    if valid_coords.size == 0:
        return {'bbox_width': np.nan, 'bbox_height': np.nan, 'bbox_diagonal': np.nan}
    min_xy = valid_coords.min(axis=0)
    max_xy = valid_coords.max(axis=0)
    width, height = (max_xy - min_xy).tolist()
    diagonal = float(np.sqrt(width ** 2 + height ** 2))
    return {'bbox_width': float(width), 'bbox_height': float(height), 'bbox_diagonal': diagonal}


def align_shape_to_reference_masked(shape: np.ndarray, reference_shape: np.ndarray, valid_mask: np.ndarray, allow_reflection: bool = False, eps: float = 1e-8) -> np.ndarray:
    valid_mask = np.asarray(valid_mask, dtype=bool)
    if valid_mask.sum() < 3:
        raise ValueError('At least 3 valid landmarks are required for masked Procrustes alignment.')
    source = np.asarray(shape[valid_mask], dtype=np.float64)
    target = np.asarray(reference_shape[valid_mask], dtype=np.float64)
    source_center = source.mean(axis=0)
    target_center = target.mean(axis=0)
    source_centered = source - source_center
    target_centered = target - target_center
    source_scale = np.linalg.norm(source_centered)
    target_scale = np.linalg.norm(target_centered)
    if source_scale <= eps or target_scale <= eps:
        raise ValueError('Degenerate shape encountered during masked Procrustes alignment.')
    source_normalized = source_centered / max(source_scale, eps)
    target_normalized = target_centered / max(target_scale, eps)
    covariance = source_normalized.T @ target_normalized
    u_matrix, _, vt_matrix = np.linalg.svd(covariance, full_matrices=False)
    rotation = u_matrix @ vt_matrix
    if not allow_reflection and np.linalg.det(rotation) < 0:
        correction = np.eye(2, dtype=np.float64)
        correction[-1, -1] = -1.0
        rotation = u_matrix @ correction @ vt_matrix
    scale = target_scale / max(source_scale, eps)
    translation = target_center - scale * (source_center @ rotation)
    aligned = np.full_like(shape, np.nan, dtype=np.float64)
    finite_shape_mask = np.isfinite(shape).all(axis=1)
    aligned[finite_shape_mask] = (scale * (shape[finite_shape_mask] @ rotation)) + translation
    return aligned


def slice_prior_to_landmarks(prior: dict[str, Any], num_landmarks: int) -> dict[str, Any]:
    if int(prior['num_landmarks']) == num_landmarks:
        return prior
    if num_landmarks > int(prior['num_landmarks']):
        raise ValueError(
            f'Cannot expand prior from {prior["num_landmarks"]} to {num_landmarks} landmarks.'
        )
    mean_shape = prior['mean_shape'].reshape(-1, 2)[:num_landmarks].reshape(-1)
    components = prior['components'].reshape(prior['components'].shape[0], -1, 2)[:, :num_landmarks, :].reshape(prior['components'].shape[0], -1)
    return {
        **prior,
        'mean_shape': mean_shape,
        'components': components,
        'reference_shape': prior['reference_shape'][:num_landmarks],
        'num_landmarks': int(num_landmarks),
        'shape_vector_size': int(num_landmarks * 2),
    }


def project_shape_into_prior(shape: np.ndarray, prior: dict[str, Any], valid_mask: np.ndarray) -> dict[str, Any]:
    valid_mask = np.asarray(valid_mask, dtype=bool)
    if valid_mask.shape[0] != shape.shape[0]:
        raise ValueError('valid_mask length must match the number of landmarks.')
    aligned_shape = align_shape_to_reference_masked(
        shape=shape,
        reference_shape=np.asarray(prior['reference_shape'], dtype=np.float64),
        valid_mask=valid_mask,
        allow_reflection=False,
        eps=float(1e-8),
    )
    vector = aligned_shape.reshape(-1)
    mean_vector = np.asarray(prior['mean_shape'], dtype=np.float64)
    components = np.asarray(prior['components'], dtype=np.float64)

    observed_dim_mask = np.repeat(valid_mask, 2)
    observed_dim_mask &= np.isfinite(vector)
    if observed_dim_mask.sum() < max(2, components.shape[0]):
        raise ValueError('Not enough observed dimensions to solve PCA projection.')

    observed_centered = vector[observed_dim_mask] - mean_vector[observed_dim_mask]
    observed_basis = components[:, observed_dim_mask].T
    coefficients, *_ = np.linalg.lstsq(observed_basis, observed_centered, rcond=None)
    reconstructed_vector = mean_vector + coefficients @ components
    reconstructed_shape = reconstructed_vector.reshape(-1, 2)

    observed_shape_mask = valid_mask & np.isfinite(aligned_shape).all(axis=1)
    if observed_shape_mask.sum() == 0:
        raise ValueError('No valid landmarks left after alignment for reconstruction error.')
    residual = aligned_shape[observed_shape_mask] - reconstructed_shape[observed_shape_mask]
    reconstruction_error = float(np.sqrt(np.mean(np.sum(residual ** 2, axis=1))))

    return {
        'aligned_shape': aligned_shape,
        'scores': coefficients,
        'reconstructed_shape': reconstructed_shape,
        'reconstruction_error': reconstruction_error,
        'observed_landmark_count': int(observed_shape_mask.sum()),
    }


def plot_shape(ax: plt.Axes, shape: np.ndarray, num_landmarks: int, title: str, annotate: bool = False) -> None:
    shape = np.asarray(shape, dtype=np.float64)
    for group_name, indices in LANDMARK_CONNECTIONS_72.items():
        valid_indices = [idx for idx in indices if idx < num_landmarks]
        if not valid_indices:
            continue
        coords = shape[valid_indices]
        finite_mask = np.isfinite(coords).all(axis=1)
        if finite_mask.sum() < 1:
            continue
        ax.plot(
            coords[finite_mask, 0],
            coords[finite_mask, 1],
            color=ANATOMICAL_GROUP_COLORS.get(group_name, '#333333'),
            linewidth=1.2,
            alpha=0.85,
        )
    for landmark_idx in range(num_landmarks):
        if not np.isfinite(shape[landmark_idx]).all():
            continue
        group = get_anatomical_group(landmark_idx)
        ax.scatter(
            shape[landmark_idx, 0],
            shape[landmark_idx, 1],
            s=22,
            color=ANATOMICAL_GROUP_COLORS.get(group, '#333333'),
        )
        if annotate:
            ax.text(shape[landmark_idx, 0], shape[landmark_idx, 1], str(landmark_idx + 1), fontsize=7)
    ax.set_title(title)
    ax.set_aspect('equal', adjustable='box')
    ax.invert_yaxis()
    ax.set_xlabel('x coordinate')
    ax.set_ylabel('y coordinate')

## Dataset loading

The following cell loads BabyLand72 and InfantFace samples using the configured paths.

In [18]:
babyland72_samples = load_babyland72_samples(BABYLAND72_ROOT)
infantface_samples = load_infantface_samples(INFANTFACE_ROOT)

babyland72_samples_68 = [trim_sample_to_landmarks(sample, COMPARISON_NUM_LANDMARKS) for sample in babyland72_samples]
infantface_samples_68 = [trim_sample_to_landmarks(sample, COMPARISON_NUM_LANDMARKS) for sample in infantface_samples]

print(f'Loaded BabyLand72 samples: {len(babyland72_samples)}')
print(f'Loaded InfantFace samples: {len(infantface_samples)}')

Loaded BabyLand72 samples: 311
Loaded InfantFace samples: 405


## Dataset sanity checks

This section computes dataset-level sanity checks and saves tables for:

- sample counts
- orientation counts
- coordinate ranges
- visibility distribution for BabyLand72
- number of valid landmarks per image
- malformed or suspicious sample warnings

In [19]:
def build_sample_level_table(samples: Sequence[NaturalShapeSample]) -> pd.DataFrame:
    rows = []
    for sample in samples:
        valid_mask = sample_valid_mask(sample)
        bbox_stats = compute_bbox_stats(sample.landmarks[valid_mask] if np.any(valid_mask) else sample.landmarks)
        visibility_rate = np.nan if sample.visibility is None else float(np.mean(sample.visibility == 1))
        rows.append({
            'dataset_name': sample.dataset_name,
            'image_id': sample.image_id,
            'label_path': str(sample.label_path),
            'image_path': str(sample.image_path) if sample.image_path is not None else None,
            'class_idx': sample.class_idx,
            'orientation': sample.orientation,
            'num_landmarks': sample.num_landmarks,
            'num_valid_landmarks': int(valid_mask.sum()),
            'visibility_rate': visibility_rate,
            'min_x': float(np.nanmin(sample.landmarks[:, 0])),
            'max_x': float(np.nanmax(sample.landmarks[:, 0])),
            'min_y': float(np.nanmin(sample.landmarks[:, 1])),
            'max_y': float(np.nanmax(sample.landmarks[:, 1])),
            **bbox_stats,
            'has_image': sample.image_path is not None,
        })
    return pd.DataFrame(rows)


def build_orientation_summary(sample_table: pd.DataFrame) -> pd.DataFrame:
    rows = []
    total = len(sample_table)
    for class_idx, class_name in CLASS_NAMES.items():
        subset = sample_table[sample_table['class_idx'] == class_idx]
        rows.append({
            'class_idx': class_idx,
            'orientation': class_name,
            'n_samples': int(len(subset)),
            'pct_samples': float(100.0 * len(subset) / total) if total > 0 else np.nan,
            'mean_valid_landmarks': float(subset['num_valid_landmarks'].mean()) if len(subset) > 0 else np.nan,
            'min_valid_landmarks': float(subset['num_valid_landmarks'].min()) if len(subset) > 0 else np.nan,
            'max_valid_landmarks': float(subset['num_valid_landmarks'].max()) if len(subset) > 0 else np.nan,
            'mean_visibility_rate': float(subset['visibility_rate'].mean()) if 'visibility_rate' in subset else np.nan,
            'mean_bbox_width': float(subset['bbox_width'].mean()) if len(subset) > 0 else np.nan,
            'mean_bbox_height': float(subset['bbox_height'].mean()) if len(subset) > 0 else np.nan,
            'mean_bbox_diagonal': float(subset['bbox_diagonal'].mean()) if len(subset) > 0 else np.nan,
        })
    return pd.DataFrame(rows)


def build_warning_table(samples: Sequence[NaturalShapeSample]) -> pd.DataFrame:
    rows = []
    for sample in samples:
        valid_mask = sample_valid_mask(sample)
        if valid_mask.sum() == 0:
            rows.append({'dataset_name': sample.dataset_name, 'image_id': sample.image_id, 'warning': 'zero_valid_landmarks'})
        if not np.isfinite(sample.landmarks).all():
            rows.append({'dataset_name': sample.dataset_name, 'image_id': sample.image_id, 'warning': 'non_finite_coordinates'})
        if sample.visibility is not None and np.any(~np.isin(sample.visibility, [0, 1])):
            rows.append({'dataset_name': sample.dataset_name, 'image_id': sample.image_id, 'warning': 'invalid_visibility_values'})
        if np.nanmax(sample.landmarks[:, 0]) - np.nanmin(sample.landmarks[:, 0]) <= 0:
            rows.append({'dataset_name': sample.dataset_name, 'image_id': sample.image_id, 'warning': 'zero_width_bbox'})
        if np.nanmax(sample.landmarks[:, 1]) - np.nanmin(sample.landmarks[:, 1]) <= 0:
            rows.append({'dataset_name': sample.dataset_name, 'image_id': sample.image_id, 'warning': 'zero_height_bbox'})
    return pd.DataFrame(rows)


baby_sample_table = build_sample_level_table(babyland72_samples)
infant_sample_table = build_sample_level_table(infantface_samples)
baby_orientation_summary = build_orientation_summary(baby_sample_table)
infant_orientation_summary = build_orientation_summary(infant_sample_table)
baby_warning_table = build_warning_table(babyland72_samples)
infant_warning_table = build_warning_table(infantface_samples)

save_table(baby_sample_table, 'babyland72_sample_level_summary.csv')
save_table(infant_sample_table, 'infantface_sample_level_summary.csv')
save_table(baby_orientation_summary, 'babyland72_orientation_summary.csv')
save_table(infant_orientation_summary, 'infantface_orientation_summary.csv')
save_table(baby_warning_table, 'babyland72_warnings.csv')
save_table(infant_warning_table, 'infantface_warnings.csv')

baby_orientation_summary

,class_idx,orientation,n_samples,pct_samples,mean_valid_landmarks,min_valid_landmarks,max_valid_landmarks,mean_visibility_rate,mean_bbox_width,mean_bbox_height,mean_bbox_diagonal
0,0,left,81,26.045016,39.716049,17.0,53.0,0.551612,0.270027,0.361569,0.455826
1,1,quarter_left,31,9.967846,59.451613,42.0,67.0,0.825717,0.310672,0.365513,0.485670
2,2,frontal,83,26.688103,59.096386,38.0,66.0,0.820783,0.355811,0.357320,0.514021
3,3,quarter_right,23,7.395498,58.478261,37.0,65.0,0.812198,0.310404,0.353157,0.478631
4,4,right,93,29.903537,39.215054,14.0,59.0,0.544654,0.276066,0.355965,0.457548


## Orientation distribution

This section creates publication-oriented orientation-distribution plots for BabyLand72 and InfantFace.

In [20]:
def plot_orientation_distributions(baby_summary: pd.DataFrame, infant_summary: pd.DataFrame) -> None:
    combined = pd.concat([
        baby_summary.assign(dataset_name='BabyLand72'),
        infant_summary.assign(dataset_name='InfantFace'),
    ], ignore_index=True)

    # Absolute counts by dataset
    fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
    x = np.arange(len(CLASS_ORDER))
    width = 0.38
    baby_counts = [int(baby_summary.loc[baby_summary['orientation'] == orientation, 'n_samples'].iloc[0]) for orientation in CLASS_ORDER]
    infant_counts = [int(infant_summary.loc[infant_summary['orientation'] == orientation, 'n_samples'].iloc[0]) for orientation in CLASS_ORDER]
    ax.bar(x - width / 2, baby_counts, width=width, color=DATASET_COLORS['BabyLand72'], label='BabyLand72')
    ax.bar(x + width / 2, infant_counts, width=width, color=DATASET_COLORS['InfantFace'], label='InfantFace')
    ax.set_title('Orientation sample counts by dataset')
    ax.set_xlabel('Orientation')
    ax.set_ylabel('Number of samples')
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_ORDER, rotation=25, ha='right')
    ax.legend(frameon=True)
    save_figure(fig, 'orientation/orientation_counts_comparison')
    plt.show()

    # Percentage distribution
    fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
    baby_pct = [float(baby_summary.loc[baby_summary['orientation'] == orientation, 'pct_samples'].iloc[0]) for orientation in CLASS_ORDER]
    infant_pct = [float(infant_summary.loc[infant_summary['orientation'] == orientation, 'pct_samples'].iloc[0]) for orientation in CLASS_ORDER]
    ax.plot(CLASS_ORDER, baby_pct, marker='o', linewidth=2.0, color=DATASET_COLORS['BabyLand72'], label='BabyLand72')
    ax.plot(CLASS_ORDER, infant_pct, marker='o', linewidth=2.0, color=DATASET_COLORS['InfantFace'], label='InfantFace')
    ax.set_title('Orientation percentage distribution by dataset')
    ax.set_xlabel('Orientation')
    ax.set_ylabel('Percentage of dataset (%)')
    ax.tick_params(axis='x', rotation=25)
    ax.legend(frameon=True)
    save_figure(fig, 'orientation/orientation_percentage_line_comparison')
    plt.show()

    # Stacked percentage bars
    fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
    bottom = np.zeros(2, dtype=float)
    dataset_names = ['BabyLand72', 'InfantFace']
    for orientation in CLASS_ORDER:
        values = []
        for dataset_name, summary_df in [('BabyLand72', baby_summary), ('InfantFace', infant_summary)]:
            values.append(float(summary_df.loc[summary_df['orientation'] == orientation, 'pct_samples'].iloc[0]))
        ax.bar(dataset_names, values, bottom=bottom, color=CLASS_COLORS[orientation], label=orientation)
        bottom += np.asarray(values)
    ax.set_title('Stacked orientation percentage distribution')
    ax.set_ylabel('Percentage of dataset (%)')
    ax.legend(title='Orientation', bbox_to_anchor=(1.02, 1), loc='upper left')
    save_figure(fig, 'orientation/orientation_percentage_stacked')
    plt.show()


plot_orientation_distributions(baby_orientation_summary, infant_orientation_summary)

/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/4067328497.py:22: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/4067328497.py:36: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/4067328497.py:52: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Mean shape analysis

This section computes:

- mean shape per dataset
- mean shape per orientation
- BabyLand72 vs InfantFace mean-shape comparison using the first 68 landmarks
- optional BabyLand72-only 72-point mean shape analysis

For BabyLand72, landmarks with `gt_visibility = 0` are excluded from geometric averaging because their coordinates are not valid.

In [21]:
def compute_mean_shape(samples: Sequence[NaturalShapeSample], num_landmarks: int) -> tuple[np.ndarray, np.ndarray]:
    coords_stack = []
    valid_stack = []
    for sample in samples:
        trimmed = trim_sample_to_landmarks(sample, num_landmarks)
        coords_stack.append(trimmed.landmarks)
        valid_stack.append(sample_valid_mask(trimmed, num_landmarks))
    coords_array = np.stack(coords_stack, axis=0).astype(np.float64)
    valid_array = np.stack(valid_stack, axis=0)
    mean_shape = np.full((num_landmarks, 2), np.nan, dtype=np.float64)
    valid_counts = valid_array.sum(axis=0)
    for landmark_idx in range(num_landmarks):
        mask = valid_array[:, landmark_idx]
        if np.any(mask):
            mean_shape[landmark_idx] = coords_array[mask, landmark_idx, :].mean(axis=0)
    return mean_shape, valid_counts


def plot_mean_shapes_by_orientation(samples: Sequence[NaturalShapeSample], dataset_name: str, num_landmarks: int, figure_stem_prefix: str) -> pd.DataFrame:
    rows = []
    fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
    axes = axes.flatten()
    for axis_index, orientation in enumerate(CLASS_ORDER):
        ax = axes[axis_index]
        subset = [trim_sample_to_landmarks(sample, num_landmarks) for sample in samples if sample.orientation == orientation]
        if not subset:
            ax.axis('off')
            continue
        mean_shape, valid_counts = compute_mean_shape(subset, num_landmarks)
        plot_shape(ax, mean_shape, num_landmarks, f'{dataset_name}: {orientation}')
        rows.append({
            'dataset_name': dataset_name,
            'orientation': orientation,
            'num_landmarks': num_landmarks,
            'n_samples': len(subset),
            'mean_valid_landmarks_per_landmark': float(np.nanmean(valid_counts)),
        })
    axes[-1].axis('off')
    fig.suptitle(f'Mean shape by orientation - {dataset_name} ({num_landmarks} landmarks)')
    save_figure(fig, f'mean_shapes/{figure_stem_prefix}_by_orientation_{num_landmarks}lm')
    plt.show()
    return pd.DataFrame(rows)


baby_mean_72, baby_counts_72 = compute_mean_shape(babyland72_samples, BABYLAND72_NUM_LANDMARKS)
baby_mean_68, baby_counts_68 = compute_mean_shape(babyland72_samples_68, COMPARISON_NUM_LANDMARKS)
infant_mean_68, infant_counts_68 = compute_mean_shape(infantface_samples_68, COMPARISON_NUM_LANDMARKS)

fig, axes = plt.subplots(1, 3 if ENABLE_72_POINT_BABYLAND72_ANALYSIS else 2, figsize=(15, 5), constrained_layout=True)
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])
plot_shape(axes[0], baby_mean_68, COMPARISON_NUM_LANDMARKS, 'BabyLand72 mean shape (68 landmarks)')
plot_shape(axes[1], infant_mean_68, COMPARISON_NUM_LANDMARKS, 'InfantFace mean shape (68 landmarks)')
if ENABLE_72_POINT_BABYLAND72_ANALYSIS:
    plot_shape(axes[2], baby_mean_72, BABYLAND72_NUM_LANDMARKS, 'BabyLand72 mean shape (72 landmarks)')
fig.suptitle('Dataset mean-shape comparison')
save_figure(fig, 'mean_shapes/dataset_mean_shape_comparison')
plt.show()

baby_mean_orientation_summary_68 = plot_mean_shapes_by_orientation(babyland72_samples_68, 'BabyLand72', COMPARISON_NUM_LANDMARKS, 'babyland72')
infant_mean_orientation_summary_68 = plot_mean_shapes_by_orientation(infantface_samples_68, 'InfantFace', COMPARISON_NUM_LANDMARKS, 'infantface')

if ENABLE_72_POINT_BABYLAND72_ANALYSIS:
    baby_mean_orientation_summary_72 = plot_mean_shapes_by_orientation(babyland72_samples, 'BabyLand72', BABYLAND72_NUM_LANDMARKS, 'babyland72')
    save_table(baby_mean_orientation_summary_72, 'babyland72_mean_shape_orientation_summary_72.csv')

save_table(baby_mean_orientation_summary_68, 'babyland72_mean_shape_orientation_summary_68.csv')
save_table(infant_mean_orientation_summary_68, 'infantface_mean_shape_orientation_summary_68.csv')

/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1197062726.py:58: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1197062726.py:41: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1197062726.py:41: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1197062726.py:41: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


PosixPath('/Users/jocareher/Documents/natural_shape_analysis_outputs/tables/infantface_mean_shape_orientation_summary_68.csv')

## Landmark variability analysis

For each dataset and orientation, this section computes:

- mean x/y
- std x/y
- spatial std per landmark
- visibility rate for BabyLand72
- anatomical group / anatomical label

It also saves variability tables and generates summary plots.

In [22]:
def compute_landmark_variability(samples: Sequence[NaturalShapeSample], num_landmarks: int) -> pd.DataFrame:
    rows = []
    scopes: list[tuple[str, Optional[int], Sequence[NaturalShapeSample]]] = [('global', None, samples)]
    for class_idx, orientation in CLASS_NAMES.items():
        subset = [sample for sample in samples if sample.class_idx == class_idx]
        scopes.append((orientation, class_idx, subset))

    for scope_name, class_idx, scope_samples in scopes:
        if not scope_samples:
            continue
        coords_stack = []
        valid_stack = []
        for sample in scope_samples:
            trimmed = trim_sample_to_landmarks(sample, num_landmarks)
            coords_stack.append(trimmed.landmarks)
            valid_stack.append(sample_valid_mask(trimmed, num_landmarks))
        coords_array = np.stack(coords_stack, axis=0)
        valid_array = np.stack(valid_stack, axis=0)
        for landmark_idx in range(num_landmarks):
            mask = valid_array[:, landmark_idx]
            coords = coords_array[mask, landmark_idx, :] if np.any(mask) else np.empty((0, 2), dtype=np.float64)
            rows.append({
                'dataset_name': scope_samples[0].dataset_name,
                'scope': scope_name,
                'class_idx': class_idx,
                'landmark_idx': landmark_idx,
                'landmark_number': landmark_idx + 1,
                'anatomical_group': get_anatomical_group(landmark_idx),
                'anatomical_label': get_anatomical_label(landmark_idx),
                'n_valid_samples': int(mask.sum()),
                'mean_x': float(coords[:, 0].mean()) if len(coords) > 0 else np.nan,
                'mean_y': float(coords[:, 1].mean()) if len(coords) > 0 else np.nan,
                'std_x': float(coords[:, 0].std()) if len(coords) > 0 else np.nan,
                'std_y': float(coords[:, 1].std()) if len(coords) > 0 else np.nan,
                'spatial_std': float(np.sqrt(coords[:, 0].var() + coords[:, 1].var())) if len(coords) > 0 else np.nan,
                'visibility_rate': float(mask.mean()),
            })
    return pd.DataFrame(rows)


def plot_spatial_std_line(variability_df: pd.DataFrame, dataset_name: str, output_stem: str) -> None:
    subset = variability_df.query("dataset_name == @dataset_name and scope == 'global'").sort_values('landmark_idx')
    fig, ax = plt.subplots(figsize=(15, 4.5), constrained_layout=True)
    ax.plot(subset['landmark_number'], subset['spatial_std'], marker='o', linewidth=1.8, color=DATASET_COLORS.get(dataset_name, '#333333'))
    ax.set_title(f'Global landmark spatial std - {dataset_name}')
    ax.set_xlabel('Landmark number')
    ax.set_ylabel('Spatial std (pixels)')
    ax.set_xticks(np.arange(1, subset['landmark_number'].max() + 1, 5))
    save_figure(fig, output_stem)
    plt.show()


def plot_group_variability_bar(variability_df: pd.DataFrame, dataset_name: str, output_stem: str) -> None:
    subset = variability_df.query("dataset_name == @dataset_name and scope == 'global'")
    grouped = subset.groupby('anatomical_group', dropna=False)['spatial_std'].mean().reset_index()
    grouped = grouped.sort_values('spatial_std', ascending=False)
    fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
    ax.bar(
        grouped['anatomical_group'],
        grouped['spatial_std'],
        color=[ANATOMICAL_GROUP_COLORS.get(group, '#333333') for group in grouped['anatomical_group']],
    )
    ax.set_title(f'Mean spatial std by anatomical group - {dataset_name}')
    ax.set_xlabel('Anatomical group')
    ax.set_ylabel('Mean spatial std (pixels)')
    ax.tick_params(axis='x', rotation=30)
    save_figure(fig, output_stem)
    plt.show()


baby_variability_72 = compute_landmark_variability(babyland72_samples, BABYLAND72_NUM_LANDMARKS)
baby_variability_68 = compute_landmark_variability(babyland72_samples_68, COMPARISON_NUM_LANDMARKS)
infant_variability_68 = compute_landmark_variability(infantface_samples_68, COMPARISON_NUM_LANDMARKS)

save_table(baby_variability_72, 'babyland72_landmark_variability_72.csv')
save_table(baby_variability_68, 'babyland72_landmark_variability_68.csv')
save_table(infant_variability_68, 'infantface_landmark_variability_68.csv')

plot_spatial_std_line(baby_variability_68, 'BabyLand72', 'variability/babyland72_spatial_std_line_68')
plot_spatial_std_line(infant_variability_68, 'InfantFace', 'variability/infantface_spatial_std_line_68')
plot_group_variability_bar(baby_variability_68, 'BabyLand72', 'variability/babyland72_group_variability_68')
plot_group_variability_bar(infant_variability_68, 'InfantFace', 'variability/infantface_group_variability_68')

if ENABLE_72_POINT_BABYLAND72_ANALYSIS:
    plot_spatial_std_line(baby_variability_72, 'BabyLand72', 'variability/babyland72_spatial_std_line_72')
    plot_group_variability_bar(baby_variability_72, 'BabyLand72', 'variability/babyland72_group_variability_72')

/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/75360671.py:50: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/75360671.py:50: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/75360671.py:68: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/75360671.py:68: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/75360671.py:50: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq

## Load synthetic PCA prior

This notebook does **not** fit PCA on natural datasets. It loads a precomputed synthetic PCA prior and uses it only for projection/reconstruction diagnostics.

In [23]:
synthetic_pca_payload = load_pca_shape_prior(SYNTHETIC_PCA_PRIOR_PATH, device='cpu')
synthetic_pca_payload.keys()

dict_keys(['priors', 'alignment', 'class_mapping', 'dataset_root', 'source_split', 'image_size'])

## Synthetic PCA projection analysis

This section projects natural shapes into the synthetic PCA subspace.

Supported analyses:

- **Global synthetic PCA**, if available in the loaded payload
- **Class-conditioned synthetic PCA**, using GT `class_idx`

For BabyLand72:
- only visible landmarks are used in the geometric projection error

For InfantFace:
- all 68 landmarks are treated as valid

In [24]:
def get_available_prior_modes(payload: dict[str, Any]) -> list[str]:
    modes = []
    if 'global_prior' in payload:
        modes.append('global')
    if 'priors' in payload and isinstance(payload['priors'], dict):
        modes.append('class_conditioned')
    return modes


def project_dataset_with_synthetic_prior(
    samples: Sequence[NaturalShapeSample],
    payload: dict[str, Any],
    num_landmarks: int,
    dataset_name: str,
) -> pd.DataFrame:
    rows = []
    available_modes = get_available_prior_modes(payload)
    for sample in samples:
        trimmed = trim_sample_to_landmarks(sample, num_landmarks)
        valid_mask = sample_valid_mask(trimmed, num_landmarks)
        base_row = {
            'dataset_name': dataset_name,
            'image_id': sample.image_id,
            'class_idx': sample.class_idx,
            'orientation': sample.orientation,
            'num_landmarks': num_landmarks,
            'observed_landmarks': int(valid_mask.sum()),
        }
        if valid_mask.sum() < 3:
            rows.append({**base_row, 'prior_mode': 'invalid', 'reconstruction_error': np.nan})
            continue

        if 'global' in available_modes:
            try:
                global_prior = slice_prior_to_landmarks(payload['global_prior'], num_landmarks)
                projection = project_shape_into_prior(trimmed.landmarks, global_prior, valid_mask)
                rows.append({
                    **base_row,
                    'prior_mode': 'global',
                    'reconstruction_error': projection['reconstruction_error'],
                    'scores': projection['scores'].tolist(),
                    'observed_landmark_count': projection['observed_landmark_count'],
                })
            except Exception as error:
                rows.append({**base_row, 'prior_mode': 'global', 'reconstruction_error': np.nan, 'error': str(error)})

        if 'class_conditioned' in available_modes:
            try:
                current_prior = payload['priors'][int(sample.class_idx)]
                current_prior = slice_prior_to_landmarks(current_prior, num_landmarks)
                projection = project_shape_into_prior(trimmed.landmarks, current_prior, valid_mask)
                rows.append({
                    **base_row,
                    'prior_mode': 'class_conditioned',
                    'reconstruction_error': projection['reconstruction_error'],
                    'scores': projection['scores'].tolist(),
                    'observed_landmark_count': projection['observed_landmark_count'],
                })
            except Exception as error:
                rows.append({**base_row, 'prior_mode': 'class_conditioned', 'reconstruction_error': np.nan, 'error': str(error)})
    return pd.DataFrame(rows)


baby_projection_68 = project_dataset_with_synthetic_prior(
    babyland72_samples_68,
    synthetic_pca_payload,
    COMPARISON_NUM_LANDMARKS,
    'BabyLand72',
)
infant_projection_68 = project_dataset_with_synthetic_prior(
    infantface_samples_68,
    synthetic_pca_payload,
    COMPARISON_NUM_LANDMARKS,
    'InfantFace',
)
projection_results_68 = pd.concat([baby_projection_68, infant_projection_68], ignore_index=True)
save_table(projection_results_68, 'natural_projection_results_68.csv')

if ENABLE_72_POINT_BABYLAND72_ANALYSIS:
    baby_projection_72 = project_dataset_with_synthetic_prior(
        babyland72_samples,
        synthetic_pca_payload,
        BABYLAND72_NUM_LANDMARKS,
        'BabyLand72',
    )
    save_table(baby_projection_72, 'babyland72_projection_results_72.csv')

projection_results_68.head()

,dataset_name,image_id,class_idx,orientation,num_landmarks,observed_landmarks,prior_mode,reconstruction_error,scores,observed_landmark_count,error
0,BabyLand72,face_bcn_00,0,left,68,28,class_conditioned,0.006108,"[-0.12534726889248526, -0.05746174501534181, 0...",28.0,NaN
1,BabyLand72,face_bcn_01,0,left,68,29,class_conditioned,0.004910,"[-0.04492492514029336, 0.00883811057847447, -0...",29.0,NaN
2,BabyLand72,face_bcn_02,1,quarter_left,68,64,class_conditioned,0.012400,"[-0.04258075264569738, -0.035168745518127, -0....",64.0,NaN
3,BabyLand72,face_bcn_03,2,frontal,68,59,class_conditioned,0.006946,"[-0.07588544660279137, 0.10326731626997229, 0....",59.0,NaN
4,BabyLand72,face_bcn_04,4,right,68,29,class_conditioned,0.005506,"[0.030220296893674378, 0.05323322379276489, 0....",29.0,NaN


## Reconstruction error summaries and plots

This section summarizes synthetic-PCA reconstruction error by:

- dataset
- orientation
- prior mode

In [25]:
def summarize_projection_errors(projection_df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        projection_df
        .groupby(['dataset_name', 'prior_mode', 'class_idx', 'orientation'], dropna=False)
        .agg(
            n_samples=('image_id', 'count'),
            n_valid_errors=('reconstruction_error', lambda values: int(np.isfinite(values).sum())),
            mean_reconstruction_error=('reconstruction_error', 'mean'),
            median_reconstruction_error=('reconstruction_error', 'median'),
            std_reconstruction_error=('reconstruction_error', 'std'),
        )
        .reset_index()
    )
    return summary


def plot_reconstruction_error_boxplots(projection_df: pd.DataFrame, num_landmarks: int, figure_stem: str) -> None:
    for prior_mode in sorted(projection_df['prior_mode'].dropna().unique()):
        subset = projection_df[projection_df['prior_mode'] == prior_mode].copy()
        if subset.empty:
            continue
        fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
        order = [('BabyLand72', orientation) for orientation in CLASS_ORDER] + [('InfantFace', orientation) for orientation in CLASS_ORDER]
        values = []
        labels = []
        colors = []
        for dataset_name, orientation in order:
            current = subset[(subset['dataset_name'] == dataset_name) & (subset['orientation'] == orientation)]['reconstruction_error'].dropna().values
            if len(current) == 0:
                continue
            values.append(current)
            labels.append(f'{dataset_name}\n{orientation}')
            colors.append(DATASET_COLORS[dataset_name])
        bp = ax.boxplot(values, patch_artist=True, showfliers=False)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.65)
        ax.set_title(f'Synthetic PCA reconstruction error by dataset and orientation ({prior_mode}, {num_landmarks} landmarks)')
        ax.set_ylabel('RMS reconstruction error in aligned PCA space')
        ax.set_xticklabels(labels, rotation=30, ha='right')
        save_figure(fig, f'reconstruction/{figure_stem}_{prior_mode}')
        plt.show()


projection_summary_68 = summarize_projection_errors(projection_results_68)
save_table(projection_summary_68, 'natural_projection_error_summary_68.csv')
plot_reconstruction_error_boxplots(projection_results_68, COMPARISON_NUM_LANDMARKS, 'projection_error_by_dataset_orientation_68')

if ENABLE_72_POINT_BABYLAND72_ANALYSIS:
    baby_projection_summary_72 = summarize_projection_errors(baby_projection_72)
    save_table(baby_projection_summary_72, 'babyland72_projection_error_summary_72.csv')

projection_summary_68

/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/3606001149.py:42: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


,dataset_name,prior_mode,class_idx,orientation,n_samples,n_valid_errors,mean_reconstruction_error,median_reconstruction_error,std_reconstruction_error
0,BabyLand72,class_conditioned,0,left,81,81,0.009084,0.009186,0.002420
1,BabyLand72,class_conditioned,1,quarter_left,31,31,0.011078,0.010542,0.002830
2,BabyLand72,class_conditioned,2,frontal,83,83,0.009776,0.009005,0.007748
3,BabyLand72,class_conditioned,3,quarter_right,23,23,0.010668,0.010073,0.001876
4,BabyLand72,class_conditioned,4,right,93,92,0.009387,0.008412,0.005434
5,InfantFace,class_conditioned,0,left,7,7,0.019179,0.017643,0.003241
6,InfantFace,class_conditioned,1,quarter_left,124,124,0.015514,0.015415,0.002217
7,InfantFace,class_conditioned,2,frontal,157,157,0.009479,0.009304,0.001517
8,InfantFace,class_conditioned,3,quarter_right,104,104,0.014304,0.014109,0.001802
9,InfantFace,class_conditioned,4,right,13,13,0.020528,0.020037,0.003184


## PCA score-space analysis

This section visualizes natural samples in the **synthetic PCA score space**.

The notebook computes PCA scores by solving for the synthetic-basis coefficients of each projected natural shape.

In [26]:
def extract_score_matrix(projection_df: pd.DataFrame, prior_mode: str) -> pd.DataFrame:
    subset = projection_df[projection_df['prior_mode'] == prior_mode].copy()
    subset = subset[subset['scores'].notna()].copy()
    if subset.empty:
        return subset
    max_components = max(len(scores) for scores in subset['scores'])
    for component_idx in range(max_components):
        subset[f'pc_{component_idx + 1}'] = subset['scores'].apply(
            lambda scores: float(scores[component_idx]) if component_idx < len(scores) else np.nan
        )
    return subset


def plot_score_scatter(score_df: pd.DataFrame, color_by: str, output_stem_prefix: str) -> None:
    available_pcs = [column for column in score_df.columns if column.startswith('pc_')]
    if len(available_pcs) < 3:
        warnings.warn('Fewer than three PCs are available for scatter plotting.')
        return
    pairs = [(1, 2), (1, 3), (2, 3)]
    for pc_x, pc_y in pairs:
        x_col = f'pc_{pc_x}'
        y_col = f'pc_{pc_y}'
        if x_col not in score_df.columns or y_col not in score_df.columns:
            continue
        fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
        if color_by == 'dataset_name':
            categories = ['BabyLand72', 'InfantFace']
            palette = DATASET_COLORS
        else:
            categories = CLASS_ORDER
            palette = CLASS_COLORS
        for category in categories:
            subset = score_df[score_df[color_by] == category]
            if subset.empty:
                continue
            ax.scatter(
                subset[x_col], subset[y_col],
                s=20, alpha=0.65,
                label=category,
                color=palette.get(category, '#333333'),
            )
        ax.set_title(f'Synthetic PCA score scatter: {x_col} vs {y_col} ({color_by})')
        ax.set_xlabel(x_col.upper())
        ax.set_ylabel(y_col.upper())
        ax.legend(frameon=True)
        save_figure(fig, f'pca_scores/{output_stem_prefix}_{color_by}_{x_col}_{y_col}')
        plt.show()


for prior_mode in sorted(projection_results_68['prior_mode'].dropna().unique()):
    score_df = extract_score_matrix(projection_results_68, prior_mode)
    if score_df.empty:
        continue
    plot_score_scatter(score_df, 'dataset_name', f'{prior_mode}_scores_68')
    plot_score_scatter(score_df, 'orientation', f'{prior_mode}_scores_68')

/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1671102754.py:47: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1671102754.py:47: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1671102754.py:47: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1671102754.py:47: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1671102754.py:47: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/

## Normality and distribution diagnostics

These plots do **not** fit a natural PCA. They only inspect whether natural samples occupy a plausible part of the synthetic PCA score space.

In [27]:
def compute_score_statistics(score_df: pd.DataFrame, prior_mode: str) -> pd.DataFrame:
    rows = []
    pc_columns = [column for column in score_df.columns if column.startswith('pc_')]
    for dataset_name in sorted(score_df['dataset_name'].dropna().unique()):
        dataset_subset = score_df[score_df['dataset_name'] == dataset_name]
        for pc_column in pc_columns:
            values = dataset_subset[pc_column].dropna().to_numpy(dtype=np.float64)
            if len(values) == 0:
                continue
            if SCIPY_AVAILABLE and len(values) >= 8:
                skewness = float(stats.skew(values, bias=False))
                kurtosis = float(stats.kurtosis(values, fisher=True, bias=False))
            else:
                mean_value = float(np.mean(values))
                std_value = float(np.std(values)) + 1e-12
                standardized = (values - mean_value) / std_value
                skewness = float(np.mean(standardized ** 3))
                kurtosis = float(np.mean(standardized ** 4) - 3.0)
            rows.append({
                'prior_mode': prior_mode,
                'dataset_name': dataset_name,
                'pc_column': pc_column,
                'n_samples': int(len(values)),
                'mean': float(np.mean(values)),
                'std': float(np.std(values)),
                'skewness': skewness,
                'excess_kurtosis': kurtosis,
            })
    return pd.DataFrame(rows)


def plot_score_histograms(score_df: pd.DataFrame, prior_mode: str, num_pcs: int = 3) -> None:
    pc_columns = [column for column in score_df.columns if column.startswith('pc_')][:num_pcs]
    for pc_column in pc_columns:
        fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
        for dataset_name in ['BabyLand72', 'InfantFace']:
            values = score_df.loc[score_df['dataset_name'] == dataset_name, pc_column].dropna().to_numpy(dtype=np.float64)
            if len(values) == 0:
                continue
            ax.hist(values, bins=30, density=True, alpha=0.45, label=dataset_name, color=DATASET_COLORS[dataset_name])
        ax.set_title(f'Synthetic PCA score histogram: {pc_column.upper()} ({prior_mode})')
        ax.set_xlabel('PCA score')
        ax.set_ylabel('Density')
        ax.legend(frameon=True)
        save_figure(fig, f'distributions/{prior_mode}_{pc_column}_histogram')
        plt.show()

        if SCIPY_AVAILABLE:
            fig = plt.figure(figsize=(8, 4.5), constrained_layout=True)
            ax = fig.add_subplot(111)
            plotted = False
            for dataset_name in ['BabyLand72', 'InfantFace']:
                values = score_df.loc[score_df['dataset_name'] == dataset_name, pc_column].dropna().to_numpy(dtype=np.float64)
                if len(values) < 8:
                    continue
                stats.probplot(values, dist='norm', plot=ax)
                plotted = True
            if plotted:
                ax.set_title(f'Q-Q plot: {pc_column.upper()} ({prior_mode})')
                save_figure(fig, f'distributions/{prior_mode}_{pc_column}_qq')
                plt.show()
            else:
                plt.close(fig)


score_statistics_frames = []
for prior_mode in sorted(projection_results_68['prior_mode'].dropna().unique()):
    score_df = extract_score_matrix(projection_results_68, prior_mode)
    if score_df.empty:
        continue
    score_statistics_frames.append(compute_score_statistics(score_df, prior_mode))
    plot_score_histograms(score_df, prior_mode, num_pcs=min(MAX_PCS_TO_PLOT, 3))

if score_statistics_frames:
    score_statistics_df = pd.concat(score_statistics_frames, ignore_index=True)
    save_table(score_statistics_df, 'natural_synthetic_pca_score_statistics.csv')
    score_statistics_df.head()

/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1996239343.py:35: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1996239343.py:46: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1996239343.py:61: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1996239343.py:46: UserWarning: Matplotlib is currently using agg, which i

## Domain gap analysis

This section collects cross-dataset comparison summaries using:

- natural landmark variability tables
- natural reconstruction errors in synthetic PCA space
- optional synthetic train references if configured

In [28]:
def build_domain_gap_summary(
    baby_variability: pd.DataFrame,
    infant_variability: pd.DataFrame,
    projection_summary: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    for dataset_name, variability_df in [('BabyLand72', baby_variability), ('InfantFace', infant_variability)]:
        subset = variability_df.query("scope == 'global'")
        rows.append({
            'dataset_name': dataset_name,
            'mean_spatial_std': float(subset['spatial_std'].mean()),
            'median_spatial_std': float(subset['spatial_std'].median()),
            'face_contour_mean_std': float(subset.loc[subset['anatomical_group'] == 'face_contour', 'spatial_std'].mean()),
            'outer_lip_mean_std': float(subset.loc[subset['anatomical_group'] == 'outer_lip', 'spatial_std'].mean()),
            'inner_lip_mean_std': float(subset.loc[subset['anatomical_group'] == 'inner_lip', 'spatial_std'].mean()),
        })
    domain_gap_df = pd.DataFrame(rows)
    reconstruction_global = (
        projection_summary
        .groupby(['dataset_name', 'prior_mode'], dropna=False)['mean_reconstruction_error']
        .mean()
        .reset_index()
    )
    domain_gap_df = domain_gap_df.merge(reconstruction_global, how='left', on='dataset_name')
    return domain_gap_df


def plot_domain_gap_variability(baby_variability: pd.DataFrame, infant_variability: pd.DataFrame) -> None:
    baby_grouped = baby_variability.query("scope == 'global'").groupby('anatomical_group')['spatial_std'].mean().rename('BabyLand72')
    infant_grouped = infant_variability.query("scope == 'global'").groupby('anatomical_group')['spatial_std'].mean().rename('InfantFace')
    combined = pd.concat([baby_grouped, infant_grouped], axis=1).reset_index()
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    x = np.arange(len(combined))
    width = 0.38
    ax.bar(x - width / 2, combined['BabyLand72'], width=width, color=DATASET_COLORS['BabyLand72'], label='BabyLand72')
    ax.bar(x + width / 2, combined['InfantFace'], width=width, color=DATASET_COLORS['InfantFace'], label='InfantFace')
    ax.set_title('Anatomical-group variability comparison')
    ax.set_xlabel('Anatomical group')
    ax.set_ylabel('Mean spatial std (pixels)')
    ax.set_xticks(x)
    ax.set_xticklabels(combined['anatomical_group'], rotation=30, ha='right')
    ax.legend(frameon=True)
    save_figure(fig, 'domain_gap/anatomical_group_variability_comparison')
    plt.show()


domain_gap_summary = build_domain_gap_summary(baby_variability_68, infant_variability_68, projection_summary_68)
save_table(domain_gap_summary, 'domain_gap_summary_68.csv')
plot_domain_gap_variability(baby_variability_68, infant_variability_68)
domain_gap_summary

/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/3619563175.py:44: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


,dataset_name,mean_spatial_std,median_spatial_std,face_contour_mean_std,outer_lip_mean_std,inner_lip_mean_std,prior_mode,mean_reconstruction_error
0,BabyLand72,0.174551,0.166738,0.185774,0.200284,0.194755,class_conditioned,0.009998
1,InfantFace,426.363687,420.922943,441.623450,423.946097,422.805344,class_conditioned,0.015801


## Optional synthetic-train comparison

If synthetic train data or synthetic summary tables are available, this section loads them for richer domain-gap comparison.

It is safe to leave this section as-is if no synthetic reference path is configured.

In [31]:
synthetic_reference_tables = {}
if SYNTHETIC_REFERENCE_TABLES_DIR is not None and SYNTHETIC_REFERENCE_TABLES_DIR.exists():
    for csv_path in sorted(SYNTHETIC_REFERENCE_TABLES_DIR.glob('*.csv')):
        try:
            synthetic_reference_tables[csv_path.stem] = pd.read_csv(csv_path)
        except Exception as error:
            warnings.warn(f'Could not load synthetic reference table {csv_path.name}: {error}')

print(f'Loaded synthetic reference tables: {sorted(synthetic_reference_tables)}')

Loaded synthetic reference tables: ['coordinate_sanity', 'dataset_summary_by_class', 'landmark_group_mapping', 'landmark_variability', 'pairwise_class_distances', 'pca_explained_variance', 'pca_reconstruction_error_summary_by_class', 'pca_reconstruction_errors', 'pca_score_statistics', 'visibility_rate_by_landmark_and_class']


## Outlier analysis

This section identifies the natural samples with the largest synthetic-PCA reconstruction error and saves:

- CSV tables
- landmark-shape plots
- optional image overlays when an image is available

In [32]:
def save_outlier_plots(
    samples: Sequence[NaturalShapeSample],
    projection_df: pd.DataFrame,
    prior_mode: str,
    num_landmarks: int,
    output_dir: Path,
    top_k: int = TOP_K_OUTLIERS,
) -> pd.DataFrame:
    output_dir.mkdir(parents=True, exist_ok=True)
    subset = projection_df[projection_df['prior_mode'] == prior_mode].copy()
    subset = subset.sort_values('reconstruction_error', ascending=False).head(top_k)
    sample_lookup = {sample.image_id: trim_sample_to_landmarks(sample, num_landmarks) for sample in samples}
    for _, row in subset.iterrows():
        sample = sample_lookup.get(row['image_id'])
        if sample is None:
            continue
        fig, ax = plt.subplots(figsize=(4.5, 4.5), constrained_layout=True)
        plot_shape(
            ax,
            sample.landmarks,
            num_landmarks,
            f"{sample.dataset_name}: {sample.image_id}\n{sample.orientation}, error={row['reconstruction_error']:.4f}",
        )
        save_figure(fig, f'outliers/{sample.dataset_name}_{prior_mode}_{sample.image_id}_{num_landmarks}lm_shape')
        plt.show()

        if sample.image_path is not None and sample.image_path.exists():
            try:
                image = np.asarray(Image.open(sample.image_path).convert('RGB'))
                fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
                ax.imshow(image)
                valid_mask = sample_valid_mask(sample, num_landmarks)
                coords = sample.landmarks[valid_mask]
                ax.scatter(coords[:, 0], coords[:, 1], s=18, c='#ffd400')
                ax.set_title(f"{sample.dataset_name}: {sample.image_id}\nLandmark overlay")
                ax.axis('off')
                save_figure(fig, f'outliers/{sample.dataset_name}_{prior_mode}_{sample.image_id}_{num_landmarks}lm_overlay')
                plt.show()
            except Exception as error:
                warnings.warn(f'Could not render overlay for {sample.image_id}: {error}')
    return subset


outlier_tables = []
for prior_mode in sorted(projection_results_68['prior_mode'].dropna().unique()):
    baby_outliers = save_outlier_plots(
        babyland72_samples_68,
        baby_projection_68,
        prior_mode,
        COMPARISON_NUM_LANDMARKS,
        OUTLIERS_DIR / f'babyland72_{prior_mode}_68',
    )
    infant_outliers = save_outlier_plots(
        infantface_samples_68,
        infant_projection_68,
        prior_mode,
        COMPARISON_NUM_LANDMARKS,
        OUTLIERS_DIR / f'infantface_{prior_mode}_68',
    )
    if not baby_outliers.empty:
        outlier_tables.append(baby_outliers)
    if not infant_outliers.empty:
        outlier_tables.append(infant_outliers)

if outlier_tables:
    top_outliers_df = pd.concat(outlier_tables, ignore_index=True)
    save_table(top_outliers_df, 'top_natural_pca_outliers_68.csv')
    top_outliers_df.head()

/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1576043965.py:25: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1576043965.py:38: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1576043965.py:25: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1576043965.py:38: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/folders/vq/s72bqs6n3xdb85wdvyfzlnwc0000gn/T/ipykernel_55762/1576043965.py:25: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/var/

## Summary section

This final cell builds a compact summary table and a markdown-style interpretation template that you can reuse in reports or manuscripts.

In [33]:
def build_final_summary_table() -> pd.DataFrame:
    rows = []
    for dataset_name, sample_table in [('BabyLand72', baby_sample_table), ('InfantFace', infant_sample_table)]:
        rows.append({
            'dataset_name': dataset_name,
            'n_samples': int(len(sample_table)),
            'mean_valid_landmarks': float(sample_table['num_valid_landmarks'].mean()),
            'mean_bbox_diagonal': float(sample_table['bbox_diagonal'].mean()),
        })
    return pd.DataFrame(rows)


final_summary_df = build_final_summary_table()
save_table(final_summary_df, 'final_dataset_summary.csv')
final_summary_df

,dataset_name,n_samples,mean_valid_landmarks,mean_bbox_diagonal
0,BabyLand72,311,48.093248,0.476533
1,InfantFace,405,68.000000,537.762802


### Interpretation template

Use the generated tables and figures to answer the following questions:

- How many samples are available in BabyLand72 and InfantFace?
- How balanced are the orientation distributions?
- Which landmarks and anatomical groups are most variable in each natural dataset?
- Which orientations have the highest synthetic-PCA reconstruction error?
- Does class-conditioned synthetic PCA represent natural data better than global synthetic PCA, when both are available?
- Is the face contour more variable in natural data than expected from synthetic references?
- Do InfantFace and BabyLand72 occupy different regions of the synthetic PCA score space?
- Which samples are the most out-of-domain according to synthetic reconstruction error?

Recommended reporting items:

- Dataset size and orientation distribution
- Landmark and anatomical-group variability summaries
- Synthetic-PCA reconstruction error by dataset and orientation
- Top outlier examples and qualitative interpretation notes